In [1]:
!pip install streamlit pyngrok fuzzywuzzy scikit-learn pandas


In [2]:
from google.colab import drive
import pandas as pd

# Mount your Google Drive
drive.mount('/content/drive')

# Update this path if your CSV is in a folder inside MyDrive
csv_path = "/content/drive/MyDrive/clean_movies.csv"

# Load dataset
movies = pd.read_csv(csv_path)
print("✅ Dataset loaded. Sample:")
movies.head()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Dataset loaded. Sample:


,title,genres
0,Toy Story (1995),Adventure Animation Children Comedy Fantasy
1,Jumanji (1995),Adventure Children Fantasy
2,Grumpier Old Men (1995),Comedy Romance
3,Waiting to Exhale (1995),Comedy Drama Romance
4,Father of the Bride Part II (1995),Comedy


In [3]:
%%writefile app.py
import streamlit as st
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from fuzzywuzzy import process

# Load dataset

csv_path = "clean_movies.csv"  # Must be in the same folder
movies = pd.read_csv(csv_path)

# Vectorize genres

tfidf = TfidfVectorizer()
count_matrix = tfidf.fit_transform(movies['genres'])

# Build Nearest Neighbors model

nn = NearestNeighbors(n_neighbors=6, metric='cosine')
nn.fit(count_matrix)

# Recommendation function

def get_recommendations(movie_title):
    # Use fuzzy matching to find closest movie title
    result = process.extractOne(movie_title, movies['title'])
    if result is None:
        return None, ["❌ Movie not found in dataset."]

    match, score = result[0], result[1]
    if score < 60:
        return None, ["❌ Movie not found in dataset."]

    idx = movies[movies['title'] == match].index[0]
    distances, indices = nn.kneighbors(count_matrix[idx])
    recommendations = [movies.iloc[i].title for i in indices[0][1:]]  # skip itself
    return match, recommendations

# Streamlit UI

st.title("🎬 Movie Recommendation System")
movie_input = st.text_input("Enter a movie title:")

if movie_input:
    matched_title, recs = get_recommendations(movie_input)
    if matched_title:
        st.success(f"Your search '{movie_input}' matched to: **{matched_title}**")
        st.subheader(f"🎬 Movies similar to '{matched_title}':")
        for r in recs:
            st.write("➡️", r)
    else:
        for msg in recs:
            st.error(msg)



Overwriting app.py


In [4]:
!cp "/content/drive/MyDrive/clean_movies.csv" "./clean_movies.csv"


In [5]:
from pyngrok import ngrok

ngrok.set_auth_token("34Z2HyYjuVZB2ysgJs7shmRJBLA_5aBaDALGHnGcFLJ7CD4yY")


In [6]:
# Open ngrok tunnel for Streamlit
public_url = ngrok.connect(addr=8501, proto="http")
print("🚀 Streamlit app URL:", public_url)

# Run Streamlit in background
!streamlit run app.py &


🚀 Streamlit app URL: NgrokTunnel: "https://paleozoulogical-stevie-unfrustratable.ngrok-free.dev" -> "http://localhost:8501"



  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.125.108.49:8501

/usr/local/lib/python3.12/dist-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')
  Stopping...
